In [ ]:
import os
from typing import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import json
import re


class article_state(TypedDict):
    origin_content: Dict[str, Any]
    messages: List[Dict[str, Any]]
    revision_count: int
    scores: Dict[str, float]
    scroes_status: int
    modify_article: Optional[Dict[str, Any]]
    final_article: Optional[Dict[str, Any]]
    suggestions: Optional[Dict[str, Any]]

In [ ]:
model = ChatOpenAI(
    model="deepseek-chat",
    api_key="your-api-key",
    base_url="llm-url",
    temperature=0.5,
    max_tokens=1024*16
)

def read_article(state: article_state):
    content = state['origin_content']
    quantity = len(content)
    print("User wants to input the content of the article with {} characters".format(quantity))
    return {}

def modify_article(state: dict):
    content = state.get('modify_article', "")
    if not content:
        origin_data = state.get('origin_content', {})
        if isinstance(origin_data, dict):
            content = origin_data.get('content', "")
        else:
            content = origin_data
    suggestions = state.get('suggestions', "").strip()
    current_count = state.get('revision_count', 0)
    suggestion_block = ""
    if suggestions:
        suggestion_block = f"""
Reviewer Suggestions:
{suggestions}

PRIMARY DIRECTIVE: You MUST carefully address and incorporate the 'Reviewer Suggestions' provided above into your revision. Use these suggestions to guide your structural and content changes.
"""
    prompt = f"""
You are a professional astrophysicist and an experienced academic editor specializing in high-impact journals such as The Astrophysical Journal (ApJ) and Monthly Notices of the Royal Astronomical Society (MNRAS).

Your task is to revise and polish the following scientific text.
{suggestion_block}

Article Content:
{content}

Requirements:
1. Logical Clarity: Improve the logical flow, coherence, and structure of the argument. Ensure that each sentence connects smoothly to the next.
2. Scientific Accuracy: Preserve the original scientific meaning while ensuring correctness and precision in terminology and interpretation.
3. Grammar and Tense: Use proper academic English grammar and appropriate tense (e.g., present tense for established facts, past tense for methods and results).
4. Academic Style: Adopt a formal, concise, and professional tone consistent with astrophysical literature. Avoid informal expressions, redundancy, and ambiguity.
5. Terminology: Use standard astrophysical terminology and conventions.
6. Conciseness: Eliminate unnecessary words while retaining all essential information.
7. Format Preservation: Keep the output in the same format as the input. For example, if the input is LaTeX, the output should also be valid LaTeX with all commands and environments preserved.

Output:
- ONLY output the revised text in the exact same format as the input.
- DO NOT output any introductory greetings, concluding remarks, explanations of your revisions, markdown code blocks (unless the original input had them), or conversational filler. Your entire response must consist solely of the polished article content.
- ⚠️ CRITICAL: You MUST output the ENTIRE manuscript from \documentclass to \end{{document}}. DO NOT skip any sections, DO NOT truncate the text, and DO NOT use placeholders like "[rest of the text omitted]"
"""
    messages = [HumanMessage(content=prompt)]
    response = model.invoke(messages)
    return {'modify_article': response.content, 'revision_count': current_count + 1}

def evaluate_article(state: article_state):
    content = state['modify_article']
    prompt = f"""
You are an expert academic editor and a professional astrophysicist with extensive experience proofreading and reviewing manuscripts for high-impact journals such as The Astrophysical Journal (ApJ) and Monthly Notices of the Royal Astronomical Society (MNRAS).

Your task is to critically evaluate the ENGLISH WRITING QUALITY, clarity, flow, and academic style of the following scientific manuscript, provide a comprehensive language review report, and assign quantitative scores to each section.

Article Content:
{content}

Evaluation Criteria for English Writing (Score each section out of 20 points, total 100 points):
1. Abstract (0-20 pts): Is the language concise, grammatically flawless, and clear? Does it articulate the core problem and results without awkward phrasing or redundancy?
2. Introduction (0-20 pts): Do the sentences and paragraphs flow logically and smoothly? Is the academic vocabulary appropriate, rich, and naturally used to frame the scientific motivation?
3. Methods (0-20 pts): Are the technical procedures and analyses described with absolute clarity and precision? Is there a consistent and correct use of tenses (e.g., past tense for procedures)?
4. Discussion (0-20 pts): Is the argumentation articulated logically and persuasively? Are complex astrophysical concepts expressed clearly without convoluted run-on sentences or ambiguity?
5. Conclusion (0-20 pts): Is the final summary written with impactful, precise, and formal academic language? Is it entirely free of grammatical errors, typos, and informal expressions?

Output Requirements:
- ONLY output the evaluation. DO NOT output any conversational filler, greetings, or rewritten text.
- You MUST start your response EXACTLY with the following structured block:

[SCORES]
abstract_score: <score>
introduction_score: <score>
methods_score: <score>
discussion_score: <score>
conclusion_score: <score>
total_score: <score>
[/SCORES]

- Follow the [SCORES] block with structured feedback, strictly divided into "Major Language Comments" and "Minor Language Comments".
- Provide specific, actionable suggestions for improving the English writing, fixing grammatical errors, enhancing sentence structure, and refining the academic tone. DO NOT critique the scientific data or physics models.
"""
    messages = [HumanMessage(content=prompt)]
    response = model.invoke(messages)
    review_text = response.content
    scores = {}
    score_block_match = re.search(r'\[SCORES\](.*?)\[/SCORES\]', review_text, re.DOTALL)
    if score_block_match:
        score_lines = score_block_match.group(1).strip().split('\n')
        for line in score_lines:
            if ':' in line:
                key, value = line.split(':')
                scores[key.strip()] = int(value.strip())
    print("Extracted scores:", scores)
    return {"evaluation": response.content, "scores": scores, "suggestions": review_text}

def judge_article(state: dict) -> str:
    scores = state.get('scores', {})
    if (scores.get('total_score', 0) < 90 or scores.get('abstract_score', 0) < 15 or scores.get('introduction_score', 0) < 15 or scores.get('methods_score', 0) < 15 or scores.get('discussion_score', 0) < 15 or scores.get('conclusion_score', 0) < 15):
        print("[Review Decision] Article not qualified and needs rewrite.")
        score_state = 0
    else:
        print("[Review Decision] Article quality is excellent, proceed to final formatting or end.")
        score_state = 1
    return {"score_state": score_state}

def end_revision(state: dict) -> str:
    print("[Process End] Revision complete, preparing final formatting or end.")
    article_state = state['modify_article']
    return {"final_state": "end", "final_article": article_state}

In [ ]:
def route_based_on_score(state: dict) -> str:
    score_state = state.get("score_state", 0)
    revision_count = state.get("revision_count", 0)
    if revision_count >= 10:
        print("Warning: maximum revision count (10) reached, forcing end process to avoid loop.")
        return "go_to_end"
    if score_state == 0:
        return "go_to_modify"
    else:
        return "go_to_end"

In [ ]:
article_graph = StateGraph(article_state)

article_graph.add_node("read_article", read_article)
article_graph.add_node("modify_article", modify_article)
article_graph.add_node("evaluate_article", evaluate_article)
article_graph.add_node("judge_article", judge_article)
article_graph.add_node("end_revision", end_revision)

article_graph.add_edge(START, "read_article")
article_graph.add_edge("read_article", "modify_article")
article_graph.add_edge("modify_article", "evaluate_article")
article_graph.add_edge("evaluate_article", "judge_article")

article_graph.add_conditional_edges(
    "judge_article",
    route_based_on_score,
    {
        "go_to_modify": "modify_article",
        "go_to_end": "end_revision"
    }
)

article_graph.add_edge("end_revision", END)


In [ ]:
original_article = {"content":"""your article content here"""
}

In [ ]:
app = article_graph.compile()

initial_state = {
    "origin_content": original_article,
    "revision_count": 0,
    "suggestions": "",
    "scores": {}
}

print("🚀 Academic review agent workflow starting...\n")

final_state = None

for output in app.stream(initial_state):
    for node_name, state_update in output.items():
        print(f"✅ Completed node: [{node_name}]")
        if node_name == "evaluate_article":
            print(f"   📊 Current scores: {state_update.get('scores')}")
        print("----------------------------------------")
        final_state = state_update

print("\n🎉 Workflow finished successfully!")

final_latex_text = final_state.get('final_article', '')

if final_latex_text:
    output_filename = "TESS_Spider_MSP_Revised.tex"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(final_latex_text)
    print(f"\n💾 Final revised manuscript saved to: [{output_filename}]")
    print("👉 You can compile it with VS Code LaTeX extension or upload to Overleaf.")
    print("\n=== 📜 Final revised manuscript (first 1000 chars preview) ===")
    print(final_latex_text[:1000] + "\n......\n[rest of content truncated]")
else:
    print("❌ Extraction failed: no final manuscript text found; check state propagation.")

🚀 启动学术评审 Agent 工作流...

User wanna input the content of the article with 1 characters
✅ 刚刚完成了节点: 【read_article】
----------------------------------------


APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}